# Notebook 3 — MLflow Prompt Registry

This notebook demonstrates the **MLflow Prompt Registry** — versioned, tracked system prompts.

**What you'll learn:**
- Registering prompts with `mlflow.register_prompt()`
- Loading a specific prompt version in code
- Rendering prompts with variable substitution
- Comparing prompt versions and their impact on output quality
- Linking prompt versions to MLflow runs for full lineage

**Prerequisites:** Docker stack running + `uv sync` completed

In [ ]:
import sys
sys.path.insert(0, '../src')

import mlflow
from agents.config import configure_mlflow, get_ollama_base_url, get_ollama_model

configure_mlflow(experiment_name='nb-03-prompt-registry')

print('Ready. MLflow tracking URI:', mlflow.get_tracking_uri())

## 1. Register a Prompt (Version 1)

In [ ]:
PROMPT_NAME = 'tech-explainer-v1'

template_v1 = """You are a helpful technical assistant.
Explain {{topic}} clearly and concisely in 2-3 sentences."""

prompt_v1 = mlflow.register_prompt(
    name=PROMPT_NAME,
    template=template_v1,
    commit_message='Initial version: concise explainer',
    tags={'style': 'concise', 'target_audience': 'engineers'},
)

print(f'Registered: {PROMPT_NAME} version {prompt_v1.version}')

## 2. Register an Improved Version (Version 2)

In [ ]:
template_v2 = """You are a senior technical educator with 10 years of experience.

Your task: Explain {{topic}} to a software engineer.

Structure your answer as:
1. **One-line definition** (what it is)
2. **Why it matters** (practical benefit)
3. **Quick example** (concrete use case)

Be precise and avoid jargon."""

prompt_v2 = mlflow.register_prompt(
    name=PROMPT_NAME,
    template=template_v2,
    commit_message='v2: structured output with definition + why + example',
    tags={'style': 'structured', 'target_audience': 'engineers'},
)

print(f'Registered: {PROMPT_NAME} version {prompt_v2.version}')

## 3. Load and Render Prompts

In [ ]:
# Load a specific version
loaded_v1 = mlflow.load_prompt(f'prompts:/{PROMPT_NAME}/1')
loaded_v2 = mlflow.load_prompt(f'prompts:/{PROMPT_NAME}/2')

topic = 'MLflow Model Registry'

rendered_v1 = loaded_v1.format(topic=topic)
rendered_v2 = loaded_v2.format(topic=topic)

print('── Version 1 rendered prompt ──')
print(rendered_v1)
print('\n── Version 2 rendered prompt ──')
print(rendered_v2)

## 4. Compare Output Quality Across Prompt Versions

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOllama(model=get_ollama_model(), base_url=get_ollama_base_url())

topics = ['MLflow experiment tracking', 'LangGraph multi-agent pipelines', 'Ollama local inference']

for version, prompt_obj in [('v1', loaded_v1), ('v2', loaded_v2)]:
    with mlflow.start_run(run_name=f'prompt-comparison-{version}'):
        mlflow.log_param('prompt_name', PROMPT_NAME)
        mlflow.log_param('prompt_version', version)
        
        all_responses = []
        for topic in topics:
            rendered = prompt_obj.format(topic=topic)
            resp = llm.invoke([HumanMessage(content=rendered)])
            all_responses.append(resp.content)
        
        avg_length = sum(len(r.split()) for r in all_responses) / len(all_responses)
        mlflow.log_metric('avg_response_word_count', avg_length)
        
        output_text = '\n\n---\n\n'.join(
            f'**Topic:** {t}\n\n{r}' for t, r in zip(topics, all_responses)
        )
        mlflow.log_text(output_text, f'outputs_{version}.md')
        print(f'[{version}] avg word count: {avg_length:.1f}')

## 5. List All Registered Prompts

In [ ]:
client = mlflow.MlflowClient()

# Search for prompts (they are stored as registered models with 'prompt' type)
prompts = client.search_prompts()
print(f'Total registered prompts: {len(prompts)}')
for p in prompts:
    print(f'  - {p.name} ({p.latest_versions[-1].version if p.latest_versions else "no versions"} versions)')

## Summary

Open **http://localhost:5000** → **Prompt Registry** tab to see:
- All registered prompts with version history
- Diff view between versions
- Tags and commit messages
- Linked runs that used each prompt version

Key takeaway: Prompt versioning gives you the same rigor as code versioning — you can reproduce any past LLM behavior by pinning a prompt version.